In [ ]:
import wandb
import polars as pl
from pathlib import Path

In [ ]:
api = wandb.Api()

ARTIFACTS = {
    "raw-site-metadata":     {"table": "site_metadata",            "partitioned": False},
    "raw-watershed-mapping": {"table": "nldas3_watershed_mapping", "partitioned": False},
    "raw-streamflow-daily":  {"table": "streamflow_daily",         "partitioned": False},
    "raw-nldas3-forcing":    {"table": "nldas3_forcing",           "partitioned": True},
    "raw-streamflow-15min":  {"table": "streamflow_15min",         "partitioned": True},
}

dfs = {}
for artifact_name, cfg in ARTIFACTS.items():
    print(f"Downloading {artifact_name}...")
    artifact = api.artifact(f"flood-forecasting/{artifact_name}:latest")
    artifact_dir = Path(artifact.download())

    table = cfg["table"]
    if cfg["partitioned"]:
        parquet_files = sorted(artifact_dir.glob(f"{table}/{table}_*.parquet"))
        df = pl.concat([pl.read_parquet(f) for f in parquet_files])
    else:
        df = pl.read_parquet(artifact_dir / table / f"{table}.parquet")

    dfs[table] = df
    print(f"  {table}: {len(df):,} rows")

In [ ]:
import duckdb

con = duckdb.connect("../flood_forecasting.duckdb", read_only=True)

print(f"{'Table':<30} {'WandB':>15} {'DuckDB':>15} {'Match':>8}")
print("-" * 70)

for table, df in dfs.items():
    db_count = con.execute(f"SELECT COUNT(*) FROM raw.{table}").fetchone()[0]
    wandb_count = len(df)
    match = wandb_count == db_count
    print(f"{table:<30} {wandb_count:>15,} {db_count:>15,} {'OK' if match else 'MISMATCH':>8}")

con.close()